# Paper 9: Reference-Channel Artifact Removal Benchmark
## Dual-Layer EEG during Real-World Table Tennis

**Dataset**: Studnicki & Ferris (2024) — *Dual-layer EEG data during real-world
table tennis.* Data in Brief, 52, 110024. OpenNeuro ds004505.

This notebook benchmarks **reference-channel artifact removal** using three
modalities available in this dataset:

| Modality | Channels | Target artifact |
|---|---|---|
| **120 noise electrodes** (dual-layer) | Mechanically coupled, electrically isolated | Motion / cable artifact |
| **8 neck EMG** | Sternocleidomastoid + trapezius | Myogenic contamination |
| **Participant-mounted IMU** | Head / body accelerometry | Movement artifact |

For each modality we compare three cleaning approaches:

1. **TSPCA** — linear time-shifted regression (de Cheveigné & Simon, 2008)
2. **IterativeDSS + nonlinear denoiser** — mne-denoise nonlinear DSS
3. **iCanClean** — sliding-window CCA (Downey & Ferris, 2022)

Evaluation metrics:
- Scalp–reference coupling reduction
- Artifact-band power reduction
- Neural signal preservation (alpha ERD)
- ICA decomposition quality


## 1 — Imports


In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.stats import pearsonr

import mne
from mne_denoise.icanclean import ICanClean
from mne_denoise.dss import IterativeDSS
from mne_denoise.dss.denoisers import (
    KurtosisDenoiser,
    SpectrogramDenoiser,
    WienerMaskDenoiser,
)


## 2 — Configuration

Adjust these constants for your local setup. By default we use one subject
and the ball-machine **stationary serve** trials (2-bounce, non-oscillating —
the most controlled ball-machine condition available for sub-03).

In [ ]:
# ── Dataset ──
DATASET_ID = "ds004505"
DATA_DIR = Path("./data/ds004505")

# Subject to analyse (sub-03 has complete .set + .fdt and BIDS metadata)
SUBJECT = "sub-03"

# Trial condition to benchmark.
# From the events.json, trial_type values are:
#   cooperative, competitive, stationary_hit, stationary_serve,
#   moving_hit, moving_serve
# Not all subjects have all conditions. sub-03 has:
#   cooperative, competitive, stationary_serve, moving_serve
CONDITION = "stationary_serve"

# ── Event types (from BIDS events.tsv / events.json) ──
# "Subject_hit"     = participant's paddle hit the ball
# "Subject_receive" = ball machine fed OR human opponent hit the ball
# "M  1"            = sync pulse (BrainVision ↔ Cometas)
HIT_EVENT_VALUE = "Subject_hit"

# ── TSPCA ──
TSPCA_SHIFTS = list(range(-10, 11))  # ±10 samples at 250 Hz ≈ ±40 ms

# ── iCanClean ──
ICANCLEAN_WINDOW_SEC = 2.0      # sliding window (seconds)
ICANCLEAN_R2_NOISE = 0.85       # r² threshold for scalp × noise
ICANCLEAN_R2_EMG = 0.40         # r² threshold for scalp × EMG
ICANCLEAN_R2_IMU = 0.65         # r² threshold for scalp × IMU

# ── IterativeDSS ──
IDSS_N_COMPONENTS = 20           # components to extract
IDSS_CORR_THRESHOLD = 0.30      # correlation with reference to flag artifact

# ── General ──
SFREQ_TARGET = 250.0            # analysis sampling rate
EPOCH_TMIN, EPOCH_TMAX = -0.5, 1.0   # epoch window around hit events (s)

## 3 — Verify local data

The dataset is stored locally under `data/ds004505`. The **sourcedata/Merged**
directory for each subject contains the synchronized, 1 Hz high-pass filtered
data from all sensors *before* artifact cleaning — exactly what we need for
benchmarking.

In [3]:
merged_dir = DATA_DIR / "sourcedata" / "Merged" / SUBJECT
assert merged_dir.exists(), f"Data not found at {merged_dir}"
print(f"Local data verified: {merged_dir}")
for f in sorted(merged_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

Local data verified: data\ds004505\sourcedata\Merged\sub-03
  sub-03_Merged.fdt  (4455.8 MB)
  sub-03_Merged.set  (310.9 MB)


## 4 — Load data and separate channels

The Merged `.set` / `.fdt` files contain **all** synchronized channels from
four LiveAmp amplifiers and the Cometas IMU system:

| Type | Count | Description |
|---|---|---|
| **Scalp EEG** | 120 | Standard 10-5 scalp electrodes |
| **Noise electrodes** | 120 | Dual-layer, mechanically coupled / electrically isolated |
| **Neck EMG** | 8 | Sternocleidomastoid + trapezius (L/R, inferior/superior) |
| **Built-in accelerometer** | 12 | 3-axis, one per LiveAmp amplifier (×4) |
| **None/broken** | 8 | Unused noise electrode slots (excluded) |
| **Cometas IMU** | 13 | Sync pulse + paddle/machine/researcher accelerometers |

**Important**: Channels are *not* contiguous by type — EMG and accelerometer
channels are interleaved among EEG/Noise rows. We read the `chanlocs.type`
field from the `.set` file for correct separation, with a name-based fallback.

We also read the `sensor_ch` mapping from `EEG.etc.ImportEvents` to identify
which Cometas IMU belongs to the participant vs. equipment (ball machine, etc.).

In [4]:
# --- Find and load the merged EEGLAB file ---
merged_dir = DATA_DIR / "sourcedata" / "Merged" / SUBJECT
set_files = sorted(merged_dir.glob("*.set"))
if not set_files:
    raise FileNotFoundError(
        f"No .set files found in {merged_dir}. Check download."
    )
print(f"Found {len(set_files)} merged file(s):")
for f in set_files:
    print(f"  {f.name}")

# Load the first merged file (contains all trial types concatenated)
raw_all = mne.io.read_raw_eeglab(str(set_files[0]), preload=True, verbose=False)
print(f"\nLoaded: {raw_all.info['nchan']} channels, "
      f"{raw_all.n_times / raw_all.info['sfreq']:.1f} s, "
      f"sfreq = {raw_all.info['sfreq']} Hz")
print(f"Channel names (first 20): {raw_all.ch_names[:20]}")
print(f"Channel names (last 20):  {raw_all.ch_names[-20:]}")


Found 1 merged file(s):
  sub-03_Merged.set


C:\Users\snesm\AppData\Local\Temp\ipykernel_33616\3196642050.py:13: RuntimeWarning: Unknown types found, setting as type EEG:
acc: ['CGY-x', 'CGY-y', 'CGY-z', 'CWR-x', 'CWR-y', 'CWR-z', 'NGY-x', 'NGY-y', 'NGY-z', 'NWR-x', 'NWR-y', 'NWR-z']
cometas: ['Emg_1(uV):', 'Imu_2_ImuAcc :X(g):', 'Imu_2_ImuAcc :Y(g):', 'Imu_2_ImuAcc :Z(g):', 'Imu_3_ImuAcc :X(g):', 'Imu_3_ImuAcc :Y(g):', 'Imu_3_ImuAcc :Z(g):', 'Imu_4_ImuAcc :X(g):', 'Imu_4_ImuAcc :Y(g):', 'Imu_4_ImuAcc :Z(g):', 'Imu_5_ImuAcc :X(g):', 'Imu_5_ImuAcc :Y(g):', 'Imu_5_ImuAcc :Z(g):']
noise: ['N-AF3', 'N-AF4', 'N-AF7', 'N-AF8', 'N-AFF1h', 'N-AFF2h', 'N-AFF5h', 'N-AFF6h', 'N-AFp1', 'N-AFp2', 'N-AFz', 'N-C1', 'N-C2', 'N-C3', 'N-C4', 'N-C5', 'N-C6', 'N-CCP1h', 'N-CCP2h', 'N-CCP3h', 'N-CCP4h', 'N-CCP5h', 'N-CCP6h', 'N-CP1', 'N-CP2', 'N-CP3', 'N-CP4', 'N-CP5', 'N-CP6', 'N-CPP1h', 'N-CPP2h', 'N-CPP3h', 'N-CPP4h', 'N-CPP5h', 'N-CPP6h', 'N-Cz', 'N-F1', 'N-F10', 'N-F2', 'N-F3', 'N-F4', 'N-F5', 'N-F6', 'N-F7', 'N-F8', 'N-F9', 'N-FC1', 'N-FC2', 'N


Loaded: 281 channels, 7928.5 s, sfreq = 500.0 Hz
Channel names (first 20): ['Fp1', 'AFp1', 'AFz', 'AF3', 'AF7', 'AFF5h', 'AFF1h', 'F1', 'F3', 'F5', 'F7', 'F9', 'FFT9h', 'FFT7h', 'FFC5h', 'FFC3h', 'FFC1h', 'FCz', 'FC1', 'FC3']
Channel names (last 20):  ['N-TTP8h', 'N-CCP6h', 'N-CCP4h', 'N-CCP2h', 'NWR-x', 'NWR-y', 'NWR-z', 'Emg_1(uV):', 'Imu_2_ImuAcc :X(g):', 'Imu_3_ImuAcc :X(g):', 'Imu_4_ImuAcc :X(g):', 'Imu_5_ImuAcc :X(g):', 'Imu_2_ImuAcc :Y(g):', 'Imu_3_ImuAcc :Y(g):', 'Imu_4_ImuAcc :Y(g):', 'Imu_5_ImuAcc :Y(g):', 'Imu_2_ImuAcc :Z(g):', 'Imu_3_ImuAcc :Z(g):', 'Imu_4_ImuAcc :Z(g):', 'Imu_5_ImuAcc :Z(g):']


C:\Users\snesm\AppData\Local\Temp\ipykernel_33616\3196642050.py:13: RuntimeWarning: Not setting positions of 8 emg channels found in montage:
['LISCM', 'LSSCM', 'LSTrap', 'LITrap', 'RITrap', 'RISCM', 'RSSCM', 'RSTrap']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw_all = mne.io.read_raw_eeglab(str(set_files[0]), preload=True, verbose=False)


### Identify and separate channel groups

We read channel types directly from the EEGLAB  file's 
field, which contains authoritative labels set during data merging:
, , , , , .

**Important**: Channels are *not* stored in contiguous blocks by type —
EMG and accelerometer channels are interleaved with EEG/Noise channels.
Positional indexing would be incorrect; we must use the metadata.

In [ ]:
n_ch = raw_all.info["nchan"]
ch_names = raw_all.ch_names


# ── Helper: extract channel types from EEGLAB .set metadata ──────────
def extract_eeglab_channel_types(set_path):
    """Extract (name, type) pairs from EEGLAB chanlocs.type field."""
    import scipy.io as sio
    mat = sio.loadmat(str(set_path), squeeze_me=True, struct_as_record=False)
    chanlocs = mat["chanlocs"]
    return [
        (str(getattr(ch, "labels", "")), str(getattr(ch, "type", "")))
        for ch in chanlocs
    ]


def classify_channel_by_name(name):
    """Fallback: infer channel type from ds004505 naming conventions."""
    if name.startswith("N-"):
        return "Noise"
    if name.startswith("None"):
        return "None"
    if any(m in name for m in ["ISCM", "SSCM", "STrap", "ITrap"]):
        return "EMG"
    if any(name.startswith(p) for p in ["CGY", "CWR", "NGY", "NWR"]):
        return "Acc"
    if "Imu_" in name or name.startswith("Emg_"):
        return "cometas"
    return "EEG"


# ── Read channel types from .set file ────────────────────────────────
try:
    ch_meta = extract_eeglab_channel_types(set_files[0])
    assert len(ch_meta) == n_ch, (
        f"chanlocs has {len(ch_meta)} entries but Raw has {n_ch} channels"
    )
    types_from_meta = {t for _, t in ch_meta}
    if types_from_meta <= {"", "EEG"}:
        raise ValueError("type field is empty or all EEG — using name fallback")
    print(f"Channel types from EEGLAB chanlocs: {sorted(types_from_meta)}")
    source = "chanlocs.type"
except Exception as e:
    print(f"Could not read chanlocs.type ({e}) — falling back to channel names")
    ch_meta = [(name, classify_channel_by_name(name)) for name in ch_names]
    source = "name heuristic"

# ── Group channel indices by type ────────────────────────────────────
idx_scalp = [i for i, (_, t) in enumerate(ch_meta) if t == "EEG"]
idx_noise = [i for i, (_, t) in enumerate(ch_meta) if t == "Noise"]
idx_emg   = [i for i, (_, t) in enumerate(ch_meta) if t == "EMG"]
idx_accel = [i for i, (_, t) in enumerate(ch_meta) if t == "Acc"]
idx_none  = [i for i, (_, t) in enumerate(ch_meta) if t == "None"]
idx_cometas = [i for i, (_, t) in enumerate(ch_meta) if t == "cometas"]

# ── Subdivide Cometas channels using sensor_ch from EEG.etc ──────────
# Per the paper, EEG.etc.ImportEvents.sensor_ch maps sensor IDs to roles:
#   synch=1, sub=2, mach=3, res=4, trig=5, net=5, head=?, torso=?, etc.
# Emg_1 is the sync pulse; Imu_N corresponds to sensor N.
import scipy.io as sio
try:
    mat = sio.loadmat(str(set_files[0]), squeeze_me=True, struct_as_record=False)
    sensor_ch = mat["etc"].ImportEvents.sensor_ch
    sensor_map = {f: getattr(sensor_ch, f)
                  for f in dir(sensor_ch) if not f.startswith("_")}
    print(f"Cometas sensor mapping: {sensor_map}")

    # Invert: sensor_number → role
    role_by_num = {str(v): k for k, v in sensor_map.items()}

    # Classify each Cometas channel by its sensor number
    idx_cometas_sync = []     # sync pulse (Emg_1)
    idx_cometas_participant = []  # participant-mounted: sub paddle, head, torso, backpack
    idx_cometas_other = []    # ball machine, researcher, net, table, trig
    participant_roles = {"sub", "head", "torso", "backpack"}

    for i in idx_cometas:
        name = ch_names[i]
        if name.startswith("Emg_"):
            idx_cometas_sync.append(i)
        elif "Imu_" in name:
            # Extract sensor number from "Imu_N_..."
            sensor_num = name.split("_")[1]
            role = role_by_num.get(sensor_num, "unknown")
            if role in participant_roles:
                idx_cometas_participant.append(i)
            else:
                idx_cometas_other.append(i)
        else:
            idx_cometas_other.append(i)
except Exception as e:
    print(f"Could not read sensor_ch ({e}) — treating all Cometas as IMU")
    idx_cometas_sync = []
    idx_cometas_participant = idx_cometas[:]
    idx_cometas_other = []

# For the IMU benchmark arm, we use participant-mounted IMUs + built-in
# accelerometers (which are on the amplifiers in the backpack)
idx_imu = idx_cometas_participant + idx_accel

# ── Set MNE channel types so downstream operations work ──────────────
mne_type_map = {}
for i in idx_noise + idx_accel + idx_none + idx_cometas:
    mne_type_map[ch_names[i]] = "misc"
for i in idx_emg:
    mne_type_map[ch_names[i]] = "emg"
if mne_type_map:
    raw_all.set_channel_types(mne_type_map)

# ── Cross-validate with BIDS _channels.tsv if available ──────────────
channels_tsv = list((DATA_DIR / SUBJECT / "eeg").glob("*_channels.tsv"))
if channels_tsv:
    import csv
    with open(channels_tsv[0], "r") as f:
        bids_ch = {row["name"]: row["type"]
                   for row in csv.DictReader(f, delimiter="\t")}
    # Verify EEG channels match
    bids_eeg = {n for n, t in bids_ch.items() if t == "EEG"}
    set_eeg = {ch_names[i] for i in idx_scalp}
    if bids_eeg == set_eeg:
        print(f"BIDS _channels.tsv validates: {len(bids_eeg)} EEG channels match")
    else:
        diff = bids_eeg.symmetric_difference(set_eeg)
        print(f"WARNING: BIDS/set EEG mismatch — {len(diff)} channels differ: {diff}")

# ── Verification summary ─────────────────────────────────────────────
def _sample_names(indices, k=3):
    names = [ch_names[i] for i in indices]
    if len(names) <= 2 * k:
        return ", ".join(names)
    return ", ".join(names[:k]) + " ... " + ", ".join(names[-k:])

print(f"\nChannel groups (source: {source}):")
print(f"  Scalp EEG:         {len(idx_scalp):>4}  [{_sample_names(idx_scalp)}]")
print(f"  Noise layer:       {len(idx_noise):>4}  [{_sample_names(idx_noise)}]")
print(f"  Neck EMG:          {len(idx_emg):>4}  [{_sample_names(idx_emg)}]")
print(f"  Accelerometer:     {len(idx_accel):>4}  [{_sample_names(idx_accel)}]")
print(f"  None/broken:       {len(idx_none):>4}  (excluded from analysis)")
print(f"  Cometas sync:      {len(idx_cometas_sync):>4}  [{_sample_names(idx_cometas_sync)}]")
print(f"  Cometas partcpnt:  {len(idx_cometas_participant):>4}  [{_sample_names(idx_cometas_participant)}]")
print(f"  Cometas other:     {len(idx_cometas_other):>4}  [{_sample_names(idx_cometas_other)}]")
print(f"  IMU (for benchmark):{len(idx_imu):>4}  (participant Cometas + built-in Acc)")
print(f"  Total:             {n_ch:>4}")
if not idx_cometas_participant:
    print("  NOTE: No participant-mounted IMUs for this subject."
          " IMU benchmark arm will use accelerometers only.")

## 5 — Data overview and sanity checks

Before benchmarking, verify:
1. Scalp channels have plausible EEG-like spectra
2. Noise channels have motion/cable-artifact-like spectra
3. EMG channels show myogenic broadband
4. IMU channels show acceleration patterns

We also check the event structure to identify hit events.


In [ ]:
# --- Downsample if needed ---
if raw_all.info["sfreq"] != SFREQ_TARGET:
    raw_all.resample(SFREQ_TARGET, verbose=False)
    print(f"Resampled to {SFREQ_TARGET} Hz")

sfreq = raw_all.info["sfreq"]

# --- Extract data arrays for each modality ---
data_all = raw_all.get_data()  # (n_ch, n_samples)
scalp_data = data_all[idx_scalp]
noise_data = data_all[idx_noise]
emg_data = data_all[idx_emg]

# For IMU: use participant-relevant Cometas IMU channels (if any)
imu_data = data_all[idx_imu_participant] if idx_imu_participant else None

print(f"Scalp: {scalp_data.shape}, Noise: {noise_data.shape}, "
      f"EMG: {emg_data.shape}, "
      f"IMU (participant): {imu_data.shape if imu_data is not None else 'N/A'}")

# --- PSD comparison across modalities ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for ax, data_block, title, color in [
    (axes[0, 0], scalp_data, f"Scalp EEG ({len(idx_scalp)} ch)", "steelblue"),
    (axes[0, 1], noise_data, f"Noise electrodes ({len(idx_noise)} ch)", "firebrick"),
    (axes[1, 0], emg_data, f"Neck EMG ({len(idx_emg)} ch)", "forestgreen"),
    (axes[1, 1],
     data_all[idx_accel] if idx_accel else emg_data,
     f"Built-in Accelerometers ({len(idx_accel)} ch)" if idx_accel else "Accel (N/A)",
     "darkorange"),
]:
    freqs_psd, psd = signal.welch(data_block, fs=sfreq, nperseg=int(2 * sfreq))
    psd_db = 10 * np.log10(psd + 1e-30)
    ax.plot(freqs_psd, psd_db.mean(axis=0), color=color, lw=1.5)
    ax.fill_between(
        freqs_psd,
        psd_db.mean(axis=0) - psd_db.std(axis=0),
        psd_db.mean(axis=0) + psd_db.std(axis=0),
        alpha=0.2, color=color,
    )
    ax.set(title=title, ylabel="PSD [dB]", xlim=(0, 100))

axes[1, 0].set_xlabel("Frequency [Hz]")
axes[1, 1].set_xlabel("Frequency [Hz]")
fig.suptitle(f"Power spectra by channel type — {SUBJECT}", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# --- Load events from BIDS events.tsv ---
# The BIDS events.tsv has richer metadata than the .set annotations:
#   value: "Subject_hit", "Subject_receive", "M  1", "boundary"
#   trial_type: "cooperative", "competitive", "stationary_serve", etc.
import csv

events_tsv_path = DATA_DIR / SUBJECT / "eeg" / f"{SUBJECT}_task-TableTennis_events.tsv"
assert events_tsv_path.exists(), (
    f"Events file not found: {events_tsv_path}\n"
    "Run: python scripts/download_ds004505.py --skip-merged"
)

with open(events_tsv_path, "r") as f:
    events_bids = list(csv.DictReader(f, delimiter="\t"))

# Count event types
from collections import Counter
value_counts = Counter(e["value"] for e in events_bids)
trial_counts = Counter(e.get("trial_type", "n/a") for e in events_bids)

print(f"Loaded {len(events_bids)} events from BIDS events.tsv")
print(f"\nEvent types (value):")
for v, c in value_counts.most_common():
    print(f"  {v:<25s} {c:>5d}")

print(f"\nTrial conditions (trial_type):")
for v, c in trial_counts.most_common():
    print(f"  {v:<25s} {c:>5d}")

# Verify our chosen CONDITION exists
condition_hits = [
    e for e in events_bids
    if e["value"] == HIT_EVENT_VALUE and e.get("trial_type") == CONDITION
]
print(f"\n{HIT_EVENT_VALUE} events in '{CONDITION}' condition: {len(condition_hits)}")
if not condition_hits:
    available = sorted({e.get("trial_type") for e in events_bids
                       if e["value"] == HIT_EVENT_VALUE} - {"STATUS"})
    print(f"WARNING: No hits in '{CONDITION}'. Available conditions: {available}")
    CONDITION = available[0]
    condition_hits = [
        e for e in events_bids
        if e["value"] == HIT_EVENT_VALUE and e.get("trial_type") == CONDITION
    ]
    print(f"Falling back to '{CONDITION}' ({len(condition_hits)} hits)")

## 6 — Epoch around hit events

We epoch around **Subject_hit** events from the BIDS `events.tsv`, filtered
by the selected `CONDITION` trial type. Each epoch gives us a short segment
of scalp + reference data for benchmarking.

Event values (from `events.json`):
- **Subject_hit**: participant's paddle made contact with the ball
- **Subject_receive**: ball machine fed OR human opponent hit the ball
- **M  1**: synchronization pulse (every 5 s, from Arduino timer)

Trial types:
- `stationary_hit` / `stationary_serve`: ball machine, no oscillation (1/2 bounces)
- `moving_hit` / `moving_serve`: ball machine with oscillation (1/2 bounces)
- `cooperative` / `competitive`: human player trials

In [ ]:
# --- Convert BIDS events to sample indices ---
# The BIDS events.tsv "onset" is in seconds, "sample" is the sample index.
# We use the sample column directly (more precise than onset * sfreq).
hit_events = [
    e for e in events_bids
    if e["value"] == HIT_EVENT_VALUE and e.get("trial_type") == CONDITION
]
print(f"Epoching {len(hit_events)} '{HIT_EVENT_VALUE}' events "
      f"in '{CONDITION}' condition")

# Convert sample indices (BIDS events use original sfreq, may need scaling)
orig_sfreq = raw_all.info["sfreq"]  # after resampling
# The BIDS sample column is at the *original* sfreq before resampling
# We need to use onset (seconds) × current sfreq for correct alignment
hit_samples = np.array([
    int(float(e["onset"]) * sfreq) for e in hit_events
])

print(f"Hit event samples (first 5): {hit_samples[:5]}")
print(f"Hit event times (first 5): {hit_samples[:5] / sfreq} s")

# Convert to sample indices for manual epoching
n_pre = int(abs(EPOCH_TMIN) * sfreq)
n_post = int(EPOCH_TMAX * sfreq)
n_epoch = n_pre + n_post

# Epoch all modalities
def epoch_data(data_2d, event_samples, n_pre, n_post):
    """Manually epoch a 2D array (n_ch, n_samples) around events."""
    epochs_list = []
    for s in event_samples:
        start = s - n_pre
        end = s + n_post
        if start >= 0 and end <= data_2d.shape[1]:
            epochs_list.append(data_2d[:, start:end])
    return np.array(epochs_list)  # (n_epochs, n_ch, n_samples)


ep_scalp = epoch_data(scalp_data, hit_samples, n_pre, n_post)
ep_noise = epoch_data(noise_data, hit_samples, n_pre, n_post)
ep_emg = epoch_data(emg_data, hit_samples, n_pre, n_post)
ep_imu = epoch_data(imu_data, hit_samples, n_pre, n_post) if imu_data is not None else None

print(f"\nEpoched shapes:")
print(f"  Scalp: {ep_scalp.shape}")
print(f"  Noise: {ep_noise.shape}")
print(f"  EMG:   {ep_emg.shape}")
if ep_imu is not None:
    print(f"  IMU:   {ep_imu.shape}")
else:
    print(f"  IMU:   N/A (no participant-mounted IMU channels)")

## 7 — Evaluation metrics

We define four complementary metrics to assess cleaning quality:

| Metric | What it measures | Good direction |
|---|---|---|
| **Scalp–reference coupling** | Mean \|r\| between scalp and reference channels | ↓ lower |
| **Artifact-band power** | Power in motion (0.5–7 Hz) or muscle (20–80 Hz) bands | ↓ lower |
| **Alpha preservation** | 8–13 Hz relative power in central/occipital channels | ≈ stable or ↑ |
| **Variance removed** | Fraction of total variance removed by cleaning | informational |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EVALUATION HELPERS
# ═══════════════════════════════════════════════════════════════════════

def scalp_ref_coupling(scalp_epochs, ref_epochs):
    """Mean absolute Pearson r between all scalp–reference channel pairs.

    Parameters
    ----------
    scalp_epochs : ndarray, shape (n_epochs, n_scalp, n_samples)
    ref_epochs : ndarray, shape (n_epochs, n_ref, n_samples)

    Returns
    -------
    float
        Mean |r| averaged over epochs and channel pairs.
    """
    n_ep = scalp_epochs.shape[0]
    r_vals = []
    # Sub-sample channel pairs for speed (max 500 pairs)
    n_scalp = scalp_epochs.shape[1]
    n_ref = ref_epochs.shape[1]
    n_pairs = min(n_scalp * n_ref, 500)
    rng = np.random.default_rng(42)
    scalp_idx = rng.integers(0, n_scalp, n_pairs)
    ref_idx = rng.integers(0, n_ref, n_pairs)
    for ep in range(min(n_ep, 20)):  # cap at 20 epochs for speed
        for si, ri in zip(scalp_idx, ref_idx):
            r = np.corrcoef(scalp_epochs[ep, si], ref_epochs[ep, ri])[0, 1]
            if np.isfinite(r):
                r_vals.append(abs(r))
    return np.mean(r_vals) if r_vals else np.nan


def band_power(epochs, sfreq, fmin, fmax):
    """Mean band power (µV²) across epochs and channels."""
    powers = []
    for ep in epochs:
        f, pxx = signal.welch(ep, fs=sfreq, nperseg=min(ep.shape[-1], int(2 * sfreq)))
        mask = (f >= fmin) & (f <= fmax)
        powers.append(pxx[:, mask].mean())
    return np.mean(powers)


def alpha_power(epochs, sfreq):
    """Relative alpha (8-13 Hz) power as fraction of 1-40 Hz total."""
    alpha = band_power(epochs, sfreq, 8, 13)
    total = band_power(epochs, sfreq, 1, 40)
    return alpha / total if total > 0 else np.nan


def variance_fraction_removed(original, cleaned):
    """Fraction of total variance removed by cleaning."""
    var_orig = np.var(original)
    var_clean = np.var(cleaned)
    return 1.0 - var_clean / var_orig if var_orig > 0 else 0.0


def evaluate_cleaning(scalp_orig, scalp_clean, ref_epochs, sfreq,
                      artifact_band=(0.5, 7.0)):
    """Run all metrics and return a results dict."""
    results = {}
    results["coupling_before"] = scalp_ref_coupling(scalp_orig, ref_epochs)
    results["coupling_after"] = scalp_ref_coupling(scalp_clean, ref_epochs)
    results["coupling_reduction"] = results["coupling_before"] - results["coupling_after"]
    results["artifact_power_before"] = band_power(scalp_orig, sfreq, *artifact_band)
    results["artifact_power_after"] = band_power(scalp_clean, sfreq, *artifact_band)
    results["artifact_power_reduction_pct"] = (
        100 * (1 - results["artifact_power_after"] / results["artifact_power_before"])
        if results["artifact_power_before"] > 0 else 0
    )
    results["alpha_before"] = alpha_power(scalp_orig, sfreq)
    results["alpha_after"] = alpha_power(scalp_clean, sfreq)
    results["variance_removed"] = variance_fraction_removed(scalp_orig, scalp_clean)
    return results


def print_results(name, results):
    """Pretty-print evaluation results."""
    print(f"\n{'─' * 55}")
    print(f"  {name}")
    print(f"{'─' * 55}")
    print(f"  Coupling |r|:  {results['coupling_before']:.4f} → "
          f"{results['coupling_after']:.4f}  "
          f"(Δ = {results['coupling_reduction']:+.4f})")
    print(f"  Artifact power: {results['artifact_power_before']:.4e} → "
          f"{results['artifact_power_after']:.4e}  "
          f"({results['artifact_power_reduction_pct']:+.1f}%)")
    print(f"  Alpha rel:     {results['alpha_before']:.4f} → "
          f"{results['alpha_after']:.4f}")
    print(f"  Var removed:   {results['variance_removed']:.2%}")


## 8 — Method implementations

### 8a — TSPCA (Time-Shift PCA / time-shifted regression)

TSPCA removes artifact by projecting scalp EEG onto a subspace spanned by
**time-shifted reference channels** and subtracting that projection
(de Cheveigné & Simon, 2008). The time shifts handle latency mismatches
between scalp and reference signals.

For EMG references, we can augment the reference matrix with **rectified /
envelope** versions to capture nonlinear coupling.


In [ ]:
def tspca_clean(scalp, ref, shifts, reg=1e-6):
    """Remove artifact via time-shifted reference regression.

    Parameters
    ----------
    scalp : ndarray, shape (n_scalp_ch, n_samples)
    ref : ndarray, shape (n_ref_ch, n_samples)
    shifts : list of int
        Lag values in samples (e.g., range(-10, 11)).
    reg : float
        Tikhonov regularization for numerical stability.

    Returns
    -------
    ndarray, shape (n_scalp_ch, n_valid_samples)
        Cleaned scalp data (trimmed by max shift on each side).
    """
    n_ref, T = ref.shape
    max_shift = max(abs(s) for s in shifts)
    T_valid = T - 2 * max_shift

    # Build time-shifted reference matrix: (T_valid, n_ref * n_shifts)
    n_shifts = len(shifts)
    R = np.empty((T_valid, n_ref * n_shifts))
    for i, s in enumerate(shifts):
        start = max_shift + s
        R[:, i * n_ref : (i + 1) * n_ref] = ref[:, start : start + T_valid].T

    # Trim scalp to valid portion
    scalp_valid = scalp[:, max_shift : max_shift + T_valid].copy()

    # Regression: scalp_clean = scalp - R @ (R^+ @ scalp)
    # Use (R^T R + λI)^{-1} R^T for stability
    RtR = R.T @ R + reg * np.eye(R.shape[1])
    coeffs = np.linalg.solve(RtR, R.T @ scalp_valid.T)
    artifact_estimate = (R @ coeffs).T

    return scalp_valid - artifact_estimate


def tspca_clean_epochs(scalp_epochs, ref_epochs, shifts, reg=1e-6):
    """Apply TSPCA to epoched data."""
    cleaned = []
    for ep_s, ep_r in zip(scalp_epochs, ref_epochs):
        c = tspca_clean(ep_s, ep_r, shifts, reg=reg)
        cleaned.append(c)
    # Align lengths (trim to shortest)
    min_len = min(c.shape[1] for c in cleaned)
    return np.array([c[:, :min_len] for c in cleaned])


def tspca_emg_augmented(scalp_epochs, emg_epochs, shifts, reg=1e-6):
    """TSPCA with rectified + envelope EMG as extra reference channels.

    Augments the EMG reference matrix with |EMG| and lowpass-envelope,
    as suggested by de Cheveigné for nonlinear artifact coupling.
    """
    augmented_refs = []
    for ep_emg in emg_epochs:
        rectified = np.abs(ep_emg)
        # Envelope via Hilbert
        envelope = np.abs(signal.hilbert(ep_emg, axis=-1))
        aug = np.vstack([ep_emg, rectified, envelope])
        augmented_refs.append(aug)
    augmented_refs = np.array(augmented_refs)
    return tspca_clean_epochs(scalp_epochs, augmented_refs, shifts, reg=reg)


### 8b — iCanClean (sliding-window CCA)

iCanClean removes EEG subspaces that are highly correlated with reference
subspaces, estimated via CCA in sliding windows (Downey & Ferris, 2022).
The original implementation used r² thresholds of 0.85 for noise electrodes
and 0.40 for neck EMG.


In [ ]:
def icanclean(scalp, ref, sfreq, window_sec=2.0, r2_threshold=0.85,
              overlap=0.0):
    """Remove artifact using ICanClean (MATLAB-verified CCA cleaning).

    Uses the mne_denoise.icanclean.ICanClean class which has been
    verified identical to MATLAB's iCanClean to machine precision.
    """
    n_scalp = scalp.shape[0]
    n_ref = ref.shape[0]
    data = np.vstack([scalp, ref])
    icc = ICanClean(
        sfreq=sfreq,
        primary_channels=list(range(n_scalp)),
        ref_channels=list(range(n_scalp, n_scalp + n_ref)),
        mode="global",
        clean_with="X",
        threshold=r2_threshold,
        segment_len=window_sec,
        overlap=overlap,
        max_reject_fraction=1.0,
        verbose=False,
    )
    cleaned = icc.fit_transform(data)
    return cleaned[:n_scalp]


def icanclean_epochs(scalp_epochs, ref_epochs, sfreq, window_sec=2.0,
                     r2_threshold=0.85, overlap=0.0):
    """Apply ICanClean to epoched data."""
    cleaned = []
    for ep_s, ep_r in zip(scalp_epochs, ref_epochs):
        c = icanclean(ep_s, ep_r, sfreq, window_sec, r2_threshold, overlap)
        cleaned.append(c)
    return np.array(cleaned)


### 8c — IterativeDSS with reference-channel guidance

We apply `IterativeDSS` from `mne-denoise` to decompose scalp EEG into
maximally non-Gaussian / non-stationary components. Then we identify which
components correlate with reference channels and remove those.

The denoiser choice depends on the artifact type:

| Reference | Denoiser | Rationale |
|---|---|---|
| Noise electrodes | `WienerMaskDenoiser` | Bursty, non-stationary motion artifact |
| Neck EMG | `KurtosisDenoiser` | Sparse, non-Gaussian muscle bursts |
| IMU | `SpectrogramDenoiser` | Time-frequency structured movement artifact |


In [ ]:
def idss_reference_clean(scalp_epochs, ref_epochs, denoiser,
                         n_components=20, corr_threshold=0.3):
    """Clean EEG using IterativeDSS + reference-channel correlation.

    1. Concatenate epochs into continuous data
    2. Fit IterativeDSS → extract components
    3. Correlate each component with reference channels
    4. Zero out artifact components → reconstruct

    Parameters
    ----------
    scalp_epochs : ndarray (n_epochs, n_scalp, n_samples)
    ref_epochs : ndarray (n_epochs, n_ref, n_samples)
    denoiser : NonlinearDenoiser instance
    n_components : int
    corr_threshold : float
        Correlation with any reference channel above this → artifact.

    Returns
    -------
    ndarray (n_epochs, n_scalp, n_samples)
        Cleaned epochs.
    """
    n_ep, n_scalp, n_time = scalp_epochs.shape
    n_ref = ref_epochs.shape[1]

    # Concatenate epochs for IterativeDSS fitting
    scalp_cat = scalp_epochs.reshape(n_ep * n_scalp, -1)  # wrong
    # Actually: concatenate along time for each channel
    scalp_cat = scalp_epochs.transpose(1, 0, 2).reshape(n_scalp, -1)  # (n_scalp, n_ep*n_time)
    ref_cat = ref_epochs.transpose(1, 0, 2).reshape(n_ref, -1)

    # Fit IterativeDSS
    n_comp = min(n_components, n_scalp)
    dss = IterativeDSS(denoiser=denoiser, n_components=n_comp, max_iter=50,
                       verbose=False)
    try:
        dss.fit(scalp_cat)
    except Exception as e:
        warnings.warn(f"IterativeDSS fit failed: {e}. Returning original data.")
        return scalp_epochs.copy()

    # Extract sources
    sources = dss.transform(scalp_cat)  # (n_comp, n_ep*n_time)
    if sources.ndim == 1:
        sources = sources[np.newaxis, :]

    # Correlate each source with reference channels
    artifact_mask = np.zeros(sources.shape[0], dtype=bool)
    for i in range(sources.shape[0]):
        max_r = 0
        for j in range(ref_cat.shape[0]):
            r = abs(np.corrcoef(sources[i], ref_cat[j])[0, 1])
            if np.isfinite(r) and r > max_r:
                max_r = r
        if max_r > corr_threshold:
            artifact_mask[i] = True

    n_removed = artifact_mask.sum()
    print(f"    IterativeDSS: {n_removed}/{sources.shape[0]} components "
          f"flagged as artifact (corr > {corr_threshold})")

    # Zero out artifact components and reconstruct
    sources_clean = sources.copy()
    sources_clean[artifact_mask] = 0
    scalp_clean_cat = dss.inverse_transform(sources_clean)

    # Reshape back to epochs
    scalp_clean = scalp_clean_cat.reshape(n_scalp, n_ep, n_time).transpose(1, 0, 2)
    return scalp_clean


## 9 — Benchmark Arm 1: Noise Electrodes

The 120 matched noise electrodes are the **primary benchmark arm** because
scalp channels correlate more with noise electrodes than with head/body
acceleration in this dataset (Studnicki et al., Sensors, 2022).

Methods:
1. **TSPCA-noise** — time-shifted regression on noise electrodes
2. **IterativeDSS + WienerMask** — bursty-artifact extraction, noise-guided
3. **iCanClean-noise** — CCA with r² = 0.85 (paper default)


In [ ]:
print("=" * 60)
print("BENCHMARK ARM 1: NOISE ELECTRODES")
print("=" * 60)

results_noise = {}

# --- 1. TSPCA-noise ---
print("\n[1/3] Running TSPCA-noise ...")
ep_scalp_tspca_noise = tspca_clean_epochs(
    ep_scalp, ep_noise, shifts=TSPCA_SHIFTS
)
# Trim reference epochs to match TSPCA output length
trim = ep_scalp.shape[-1] - ep_scalp_tspca_noise.shape[-1]
trim_l = trim // 2
ep_noise_trimmed = ep_noise[:, :, trim_l : trim_l + ep_scalp_tspca_noise.shape[-1]]
ep_scalp_trimmed = ep_scalp[:, :, trim_l : trim_l + ep_scalp_tspca_noise.shape[-1]]

results_noise["TSPCA"] = evaluate_cleaning(
    ep_scalp_trimmed, ep_scalp_tspca_noise, ep_noise_trimmed, sfreq,
    artifact_band=(0.5, 7.0),
)
print_results("TSPCA-noise", results_noise["TSPCA"])

# --- 2. IterativeDSS + WienerMask ---
print("\n[2/3] Running IterativeDSS + WienerMask (noise-guided) ...")
wiener = WienerMaskDenoiser(window_samples=int(0.2 * sfreq), noise_percentile=25.0)
ep_scalp_idss_noise = idss_reference_clean(
    ep_scalp, ep_noise, denoiser=wiener,
    n_components=IDSS_N_COMPONENTS, corr_threshold=IDSS_CORR_THRESHOLD,
)
results_noise["IterDSS+WienerMask"] = evaluate_cleaning(
    ep_scalp, ep_scalp_idss_noise, ep_noise, sfreq,
    artifact_band=(0.5, 7.0),
)
print_results("IterDSS + WienerMask (noise)", results_noise["IterDSS+WienerMask"])

# --- 3. iCanClean-noise ---
print("\n[3/3] Running iCanClean-noise ...")
ep_scalp_ican_noise = icanclean_epochs(
    ep_scalp, ep_noise, sfreq,
    window_sec=ICANCLEAN_WINDOW_SEC, r2_threshold=ICANCLEAN_R2_NOISE,
)
results_noise["iCanClean"] = evaluate_cleaning(
    ep_scalp, ep_scalp_ican_noise, ep_noise, sfreq,
    artifact_band=(0.5, 7.0),
)
print_results("iCanClean (noise, r²=0.85)", results_noise["iCanClean"])


In [ ]:
# --- Visualization: Noise electrode benchmark ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

methods = list(results_noise.keys())
colors = ["#2196F3", "#FF5722", "#4CAF50"]

# Coupling reduction
vals = [results_noise[m]["coupling_reduction"] for m in methods]
axes[0].bar(methods, vals, color=colors)
axes[0].set_ylabel("Δ |r| (coupling reduction)")
axes[0].set_title("Scalp–Noise Coupling Reduction")
axes[0].axhline(0, ls="--", color="gray", lw=0.8)

# Artifact power reduction
vals = [results_noise[m]["artifact_power_reduction_pct"] for m in methods]
axes[1].bar(methods, vals, color=colors)
axes[1].set_ylabel("Power reduction (%)")
axes[1].set_title("Low-Freq Artifact Power (0.5–7 Hz)")

# Alpha preservation
vals_before = [results_noise[m]["alpha_before"] for m in methods]
vals_after = [results_noise[m]["alpha_after"] for m in methods]
x = np.arange(len(methods))
axes[2].bar(x - 0.15, vals_before, 0.3, label="Before", color="lightgray")
axes[2].bar(x + 0.15, vals_after, 0.3, label="After", color=colors)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=15)
axes[2].set_ylabel("Relative alpha power")
axes[2].set_title("Alpha Preservation (8–13 Hz)")
axes[2].legend()

fig.suptitle("Benchmark Arm 1: Noise Electrodes", fontsize=13)
fig.tight_layout()
plt.show()


## 10 — Benchmark Arm 2: Neck EMG

The 8 neck EMG channels (sternocleidomastoid + trapezius) capture myogenic
contamination most visible at posterior/lateral scalp sites in the
high-frequency range.

Methods:
1. **TSPCA-EMG** — time-shifted regression with **rectified + envelope EMG**
   as additional reference channels (augmented refs per de Cheveigné)
2. **IterativeDSS + KurtosisDenoiser** — sparse non-Gaussian burst extraction
3. **iCanClean-EMG** — CCA with r² = 0.40 (paper default for EMG)


In [ ]:
print("=" * 60)
print("BENCHMARK ARM 2: NECK EMG")
print("=" * 60)

results_emg = {}

# --- 1. TSPCA-EMG (rectified + envelope augmented) ---
print("\n[1/3] Running TSPCA-EMG (augmented) ...")
ep_scalp_tspca_emg = tspca_emg_augmented(
    ep_scalp, ep_emg, shifts=TSPCA_SHIFTS,
)
trim_emg = ep_scalp.shape[-1] - ep_scalp_tspca_emg.shape[-1]
trim_l_emg = trim_emg // 2
ep_emg_trimmed = ep_emg[:, :, trim_l_emg : trim_l_emg + ep_scalp_tspca_emg.shape[-1]]
ep_scalp_trimmed_emg = ep_scalp[:, :, trim_l_emg : trim_l_emg + ep_scalp_tspca_emg.shape[-1]]

results_emg["TSPCA"] = evaluate_cleaning(
    ep_scalp_trimmed_emg, ep_scalp_tspca_emg, ep_emg_trimmed, sfreq,
    artifact_band=(20.0, 80.0),  # muscle band
)
print_results("TSPCA-EMG (augmented)", results_emg["TSPCA"])

# --- 2. IterativeDSS + KurtosisDenoiser ---
print("\n[2/3] Running IterativeDSS + KurtosisDenoiser (EMG-guided) ...")
kurtosis = KurtosisDenoiser(nonlinearity="tanh")
ep_scalp_idss_emg = idss_reference_clean(
    ep_scalp, ep_emg, denoiser=kurtosis,
    n_components=IDSS_N_COMPONENTS, corr_threshold=IDSS_CORR_THRESHOLD,
)
results_emg["IterDSS+Kurtosis"] = evaluate_cleaning(
    ep_scalp, ep_scalp_idss_emg, ep_emg, sfreq,
    artifact_band=(20.0, 80.0),
)
print_results("IterDSS + Kurtosis (EMG)", results_emg["IterDSS+Kurtosis"])

# --- 3. iCanClean-EMG ---
print("\n[3/3] Running iCanClean-EMG ...")
ep_scalp_ican_emg = icanclean_epochs(
    ep_scalp, ep_emg, sfreq,
    window_sec=ICANCLEAN_WINDOW_SEC, r2_threshold=ICANCLEAN_R2_EMG,
)
results_emg["iCanClean"] = evaluate_cleaning(
    ep_scalp, ep_scalp_ican_emg, ep_emg, sfreq,
    artifact_band=(20.0, 80.0),
)
print_results("iCanClean (EMG, r²=0.40)", results_emg["iCanClean"])


In [ ]:
# --- Visualization: EMG benchmark ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

methods = list(results_emg.keys())
colors = ["#2196F3", "#FF5722", "#4CAF50"]

vals = [results_emg[m]["coupling_reduction"] for m in methods]
axes[0].bar(methods, vals, color=colors)
axes[0].set_ylabel("Δ |r| (coupling reduction)")
axes[0].set_title("Scalp–EMG Coupling Reduction")
axes[0].axhline(0, ls="--", color="gray", lw=0.8)

vals = [results_emg[m]["artifact_power_reduction_pct"] for m in methods]
axes[1].bar(methods, vals, color=colors)
axes[1].set_ylabel("Power reduction (%)")
axes[1].set_title("Muscle-Band Power (20–80 Hz)")

vals_before = [results_emg[m]["alpha_before"] for m in methods]
vals_after = [results_emg[m]["alpha_after"] for m in methods]
x = np.arange(len(methods))
axes[2].bar(x - 0.15, vals_before, 0.3, label="Before", color="lightgray")
axes[2].bar(x + 0.15, vals_after, 0.3, label="After", color=colors)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods, rotation=15)
axes[2].set_ylabel("Relative alpha power")
axes[2].set_title("Alpha Preservation (8–13 Hz)")
axes[2].legend()

fig.suptitle("Benchmark Arm 2: Neck EMG", fontsize=13)
fig.tight_layout()
plt.show()


## 11 — Benchmark Arm 3: IMU / Accelerometry (exploratory)

The participant-mounted IMU/accelerometer channels are the **weakest**
reference arm: in the dual-layer characterization paper, matched noise
electrodes explained scalp contamination better than head/body acceleration.

We include this arm for completeness but treat it as exploratory.

**Available participant-mounted sensors vary by subject:**
- **sub-01 to sub-08**: Only the participant paddle IMU (`sub`) — no head/torso
- **sub-09 onwards**: Additional backpack IMU available
- All subjects have **built-in accelerometers** (12 channels in the LiveAmp
  amplifiers, which sit in the backpack on the upper back)

Methods:
1. **TSPCA-IMU** — time-shifted regression on participant IMU channels
2. **IterativeDSS + SpectrogramDenoiser** — TF-structured movement artifact

> **Note**: We use participant-mounted IMUs + built-in accelerometers.
> Paddle, ball-machine, and researcher IMUs are excluded as they carry
> task timing rather than body-movement artifact.

In [ ]:
# --- Combine accelerometer + participant IMU into movement reference ---
# Built-in LiveAmp accelerometers (always available, 12 ch)
accel_data = data_all[idx_accel] if idx_accel else None

# Participant-relevant Cometas IMU (head, torso, backpack, paddle)
imu_participant_data = data_all[idx_imu_participant] if idx_imu_participant else None

# Combine into a single movement reference matrix
movement_parts = []
movement_labels = []
if accel_data is not None and len(idx_accel) > 0:
    movement_parts.append(accel_data)
    movement_labels.append(f"Acc ({len(idx_accel)} ch)")
if imu_participant_data is not None and len(idx_imu_participant) > 0:
    movement_parts.append(imu_participant_data)
    movement_labels.append(f"Participant IMU ({len(idx_imu_participant)} ch)")

if movement_parts:
    movement_data = np.concatenate(movement_parts, axis=0)
    print("=" * 60)
    print("BENCHMARK ARM 3: IMU / ACCELEROMETRY")
    print("=" * 60)
    print(f"Movement reference: {' + '.join(movement_labels)} "
          f"= {movement_data.shape[0]} channels total")

    # Epoch movement data
    ep_movement = epoch_data(movement_data, hit_samples, n_pre, n_post)
    print(f"Movement epochs: {ep_movement.shape}")

    results_imu = {}

    # --- 1. TSPCA-IMU ---
    print("\n[1/2] Running TSPCA-IMU ...")
    ep_scalp_tspca_imu = tspca_clean_epochs(
        ep_scalp, ep_movement, shifts=TSPCA_SHIFTS,
    )
    trim_imu = ep_scalp.shape[-1] - ep_scalp_tspca_imu.shape[-1]
    trim_l_imu = trim_imu // 2
    ep_mov_trimmed = ep_movement[:, :, trim_l_imu : trim_l_imu + ep_scalp_tspca_imu.shape[-1]]
    ep_scalp_trimmed_imu = ep_scalp[:, :, trim_l_imu : trim_l_imu + ep_scalp_tspca_imu.shape[-1]]

    results_imu["TSPCA"] = evaluate_cleaning(
        ep_scalp_trimmed_imu, ep_scalp_tspca_imu, ep_mov_trimmed, sfreq,
        artifact_band=(0.5, 7.0),
    )
    print_results("TSPCA-IMU", results_imu["TSPCA"])

    # --- 2. IterativeDSS + SpectrogramDenoiser ---
    print("\n[2/2] Running IterativeDSS + SpectrogramDenoiser (IMU-guided) ...")
    spectrogram = SpectrogramDenoiser(threshold_percentile=90.0,
                                      nperseg=int(0.5 * sfreq))
    ep_scalp_idss_imu = idss_reference_clean(
        ep_scalp, ep_movement, denoiser=spectrogram,
        n_components=IDSS_N_COMPONENTS, corr_threshold=IDSS_CORR_THRESHOLD,
    )
    results_imu["IterDSS+Spectrogram"] = evaluate_cleaning(
        ep_scalp, ep_scalp_idss_imu, ep_movement, sfreq,
        artifact_band=(0.5, 7.0),
    )
    print_results("IterDSS + Spectrogram (IMU)", results_imu["IterDSS+Spectrogram"])

else:
    print("No accelerometer or participant IMU channels available — skipping Arm 3.")
    results_imu = {}

In [ ]:
# --- Visualization: IMU benchmark (if available) ---
if results_imu:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    methods = list(results_imu.keys())
    colors = ["#2196F3", "#FF5722"]

    vals = [results_imu[m]["coupling_reduction"] for m in methods]
    axes[0].bar(methods, vals, color=colors)
    axes[0].set_ylabel("Δ |r|")
    axes[0].set_title("Scalp–IMU Coupling Reduction")
    axes[0].axhline(0, ls="--", color="gray", lw=0.8)

    vals = [results_imu[m]["artifact_power_reduction_pct"] for m in methods]
    axes[1].bar(methods, vals, color=colors)
    axes[1].set_ylabel("Power reduction (%)")
    axes[1].set_title("Low-Freq Power (0.5–7 Hz)")

    vals_before = [results_imu[m]["alpha_before"] for m in methods]
    vals_after = [results_imu[m]["alpha_after"] for m in methods]
    x = np.arange(len(methods))
    axes[2].bar(x - 0.15, vals_before, 0.3, label="Before", color="lightgray")
    axes[2].bar(x + 0.15, vals_after, 0.3, label="After", color=colors)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(methods)
    axes[2].set_ylabel("Relative alpha power")
    axes[2].set_title("Alpha Preservation")
    axes[2].legend()

    fig.suptitle("Benchmark Arm 3: IMU (exploratory)", fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print("No IMU results to plot.")


## 12 — Cross-method comparison

Aggregate all benchmark results into a single summary table and radar chart.


In [ ]:
# --- Build summary table ---
all_results = {}
for name, res in results_noise.items():
    all_results[f"Noise: {name}"] = res
for name, res in results_emg.items():
    all_results[f"EMG: {name}"] = res
for name, res in results_imu.items():
    all_results[f"IMU: {name}"] = res

# Print table
header = f"{'Method':<35s} {'Coupl Δ':>8s} {'Art %':>8s} {'α pres':>8s} {'Var rm':>8s}"
print("=" * 70)
print("CROSS-METHOD COMPARISON")
print("=" * 70)
print(header)
print("-" * 70)
for name, r in all_results.items():
    print(f"{name:<35s} "
          f"{r['coupling_reduction']:>+8.4f} "
          f"{r['artifact_power_reduction_pct']:>+7.1f}% "
          f"{r['alpha_after']:>8.4f} "
          f"{r['variance_removed']:>7.1%}")
print("=" * 70)


In [ ]:
# --- Radar / spider chart ---
from matplotlib.patches import FancyBboxPatch

metrics_keys = ["coupling_reduction", "artifact_power_reduction_pct",
                "alpha_after", "variance_removed"]
metric_labels = ["Coupling\nReduction", "Artifact\nPower Reduction",
                 "Alpha\nPreservation", "Variance\nRemoved"]

# Normalize each metric to [0, 1] for radar chart
all_vals = {k: [] for k in metrics_keys}
for r in all_results.values():
    for k in metrics_keys:
        all_vals[k].append(r[k])

norm_results = {}
for name, r in all_results.items():
    norm = []
    for k in metrics_keys:
        vmin = min(all_vals[k])
        vmax = max(all_vals[k])
        if vmax - vmin > 0:
            norm.append((r[k] - vmin) / (vmax - vmin))
        else:
            norm.append(0.5)
    norm_results[name] = norm

# Plot radar
n_metrics = len(metrics_keys)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10
for i, (name, vals) in enumerate(norm_results.items()):
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, "o-", label=name, color=cmap(i), lw=2)
    ax.fill(angles, vals_plot, alpha=0.1, color=cmap(i))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_title("Cross-Method Comparison (normalized)", pad=20, fontsize=13)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.show()


## 13 — ICA quality assessment (optional)

Run ICA on original and best-cleaned data from each arm, then compare
the number of brain-like components. This is the key downstream metric
used by the Ferris group in both the dual-layer and iCanClean papers.

> **Note**: This cell is computationally expensive. Skip if you only need
> the spectral/coupling metrics above.


In [ ]:
def run_ica_and_score(epochs_data, sfreq, ch_names_scalp, label=""):
    """Run ICA on epoched scalp data and return quality metrics.

    Returns dict with n_components, explained_var_95pct_rank, etc.
    """
    # Concatenate epochs into continuous for ICA
    n_ep, n_ch, n_t = epochs_data.shape
    data_cat = epochs_data.transpose(1, 0, 2).reshape(n_ch, -1)

    # Create Raw object
    info = mne.create_info(ch_names=ch_names_scalp[:n_ch], sfreq=sfreq,
                           ch_types="eeg")
    raw_tmp = mne.io.RawArray(data_cat, info, verbose=False)

    # Determine rank
    rank = np.linalg.matrix_rank(data_cat @ data_cat.T / data_cat.shape[1])
    n_ica = min(rank - 1, 60)  # cap for speed

    if n_ica < 5:
        return {"label": label, "n_ica": n_ica, "note": "rank too low"}

    # Run ICA
    ica = mne.preprocessing.ICA(n_components=n_ica, method="fastica",
                                 random_state=42, verbose=False)
    try:
        ica.fit(raw_tmp, verbose=False)
    except Exception as e:
        return {"label": label, "error": str(e)}

    # Score: variance explained by each IC
    sources = ica.get_sources(raw_tmp).get_data()
    var_per_ic = np.var(sources, axis=1)
    var_total = np.var(data_cat)

    # Kurtosis of each IC (brain components tend to be mildly super-Gaussian)
    from scipy.stats import kurtosis as sp_kurtosis
    ic_kurtosis = sp_kurtosis(sources, axis=1)

    # Count "brain-like" ICs: moderate kurtosis (0–10), not too spiky
    n_brainlike = np.sum((ic_kurtosis > 0) & (ic_kurtosis < 10))

    results = {
        "label": label,
        "n_ica": n_ica,
        "data_rank": rank,
        "n_brainlike_ics": int(n_brainlike),
        "mean_kurtosis": float(np.mean(ic_kurtosis)),
        "median_kurtosis": float(np.median(ic_kurtosis)),
    }
    print(f"  {label}: rank={rank}, n_ICA={n_ica}, brain-like ICs={n_brainlike}, "
          f"mean kurt={np.mean(ic_kurtosis):.2f}")
    return results


# --- Run ICA on original and best-performing cleaned data ---
ch_names_scalp = [raw_all.ch_names[i] for i in idx_scalp]

print("Running ICA quality assessment ...")
print("(this may take a few minutes)\n")

ica_results = {}
ica_results["Original"] = run_ica_and_score(ep_scalp, sfreq, ch_names_scalp, "Original")

# Best noise method: use iCanClean as the paper's own method
ica_results["Noise: iCanClean"] = run_ica_and_score(
    ep_scalp_ican_noise, sfreq, ch_names_scalp, "Noise: iCanClean"
)

# Best EMG method
ica_results["EMG: iCanClean"] = run_ica_and_score(
    ep_scalp_ican_emg, sfreq, ch_names_scalp, "EMG: iCanClean"
)

print("\nICA quality summary:")
for name, r in ica_results.items():
    if "error" in r:
        print(f"  {name}: ERROR — {r['error']}")
    elif "note" in r:
        print(f"  {name}: {r['note']}")
    else:
        print(f"  {name}: {r['n_brainlike_ics']} brain-like ICs "
              f"(median kurtosis={r['median_kurtosis']:.2f})")


## 14 — Summary & recommendations

### Key findings

1. **Noise electrodes** are the strongest reference modality — they capture
   motion/cable artifact more directly than EMG or IMU.

2. **Neck EMG** targets high-frequency myogenic contamination specifically
   at posterior/lateral scalp sites.

3. **IMU** is the weakest reference arm, consistent with the original
   characterization paper finding that matched noise electrodes explain
   scalp contamination better than head/body acceleration.

### Method comparison

| Method | Best for | Strengths | Limitations |
|---|---|---|---|
| **TSPCA** | Direct artifact subtraction | Simple, fast, interpretable | Linear assumption, needs time-shift tuning |
| **IterativeDSS** | Non-stationary / non-Gaussian artifacts | Adaptive, nonlinear | Slower, correlation threshold is a free parameter |
| **iCanClean** | Paper-native comparator | Validated in this dataset | Window size and r² sensitivity |

### Recommended pipeline

For this dataset, the strongest combination is:
1. **iCanClean with noise electrodes** (motion artifact, r² ≈ 0.85)
2. **TSPCA with augmented EMG** (muscle artifact, with rectified/envelope refs)
3. Run ICA/AMICA after cleaning for source decomposition


In [ ]:
# --- Final summary table ---
print("=" * 75)
print("PAPER 9 — REFERENCE-CHANNEL BENCHMARK SUMMARY")
print(f"Subject: {SUBJECT} | Condition: {CONDITION}")
print("=" * 75)
print()

print(f"{'Arm':<8s} {'Method':<28s} {'Coupl Δ':>8s} {'Art %':>8s} {'α pres':>8s} {'Var rm':>8s}")
print("-" * 75)

for arm_name, arm_results in [("Noise", results_noise), ("EMG", results_emg),
                               ("IMU", results_imu)]:
    for method_name, r in arm_results.items():
        print(f"{arm_name:<8s} {method_name:<28s} "
              f"{r['coupling_reduction']:>+8.4f} "
              f"{r['artifact_power_reduction_pct']:>+7.1f}% "
              f"{r['alpha_after']:>8.4f} "
              f"{r['variance_removed']:>7.1%}")

print("=" * 75)
print()
print("Higher coupling reduction = better artifact removal")
print("Higher artifact power reduction = more artifact removed")
print("Alpha preservation ≈ stable = neural signal not damaged")
print("Variance removed = aggressiveness (too high → over-cleaning)")
